# PLS Grid Search - Tìm Top Model
Scan toàn bộ tổ hợp `(window_start, window_size, n_points)` để tìm bộ V features cho PLS tốt nhất.

**Output:** Bảng top models theo composite score (R² + RMSE) + visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded')

---
## CONFIG - Chinh tai day!

In [ ]:
FILE_PATH    = 'CUCOMOF.csv'

RANGE_START  = -0.2   # V bat dau scan
RANGE_END    =  0.8   # V ket thuc scan

WINDOW_MIN   =  0.3   # Do rong window nho nhat (V)
WINDOW_MAX   =  0.8   # Do rong window lon nhat (V)

STEP_MIN     =  5     # So diem toi thieu trong window
STEP_MAX     =  20    # So diem toi da trong window

WINDOW_SLIDE =  0.01  # Buoc dich window (V) - nho hon = min hon, cham hon

TOP_N        =  20    # Hien thi top N models

# Uoc tinh
n_ws  = len(np.arange(WINDOW_MIN, WINDOW_MAX + 0.001, 0.01))
n_pts = STEP_MAX - STEP_MIN + 1
n_st  = int((RANGE_END - RANGE_START - WINDOW_MIN) / WINDOW_SLIDE) + 1
print(f'Config: range=[{RANGE_START},{RANGE_END}]V  window=[{WINDOW_MIN},{WINDOW_MAX}]V  points=[{STEP_MIN},{STEP_MAX}]')
print(f'Uoc tinh combinations: ~{n_ws * n_pts * n_st:,}')
print(f'Uoc tinh thoi gian: ~{n_ws * n_pts * n_st * 0.005 / 60:.1f} phut')

---
## 1. Load Data

In [ ]:
CONCENTRATIONS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
CONC_COLS = [f'{c}mM' for c in CONCENTRATIONS]
y = np.array(CONCENTRATIONS)

data_rows = []
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line or i in [0, 1, 2]:
            continue
        parts = [p.strip() for p in line.split('\t')]
        clean = [p for p in parts if p != '']
        clean = clean[:11] if len(clean) >= 11 else clean + [np.nan]*(11-len(clean))
        data_rows.append(clean)

df_raw = pd.DataFrame(data_rows, columns=['V'] + CONC_COLS)
df_raw = df_raw.apply(pd.to_numeric, errors='coerce')
df_fwd = df_raw.iloc[:df_raw['V'].idxmax()+1].copy().reset_index(drop=True)

df_scan = df_fwd[(df_fwd['V'] >= RANGE_START - 0.005) & (df_fwd['V'] <= RANGE_END + 0.005)].copy()
V_array  = df_scan['V'].values
I_matrix = df_scan[CONC_COLS].values  # (n_V, 10)

print(f'Data: {len(df_scan)} V points  [{V_array.min():.3f}V -> {V_array.max():.3f}V]')

---
## 2. Grid Search

In [ ]:
def eval_pls_loo(X, y):
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    loo = LeaveOneOut()
    best_r2, best_rmse, best_nc = -999, 999, 1
    for nc in range(1, min(X.shape[1], len(y)-1) + 1):
        pls = PLSRegression(n_components=nc)
        yp  = cross_val_predict(pls, X_s, y, cv=loo).ravel()
        r2  = r2_score(y, yp)
        rmse = np.sqrt(mean_squared_error(y, yp))
        if r2 > best_r2:
            best_r2, best_rmse, best_nc = r2, rmse, nc
    return best_r2, best_rmse, best_nc

def composite_score(r2, rmse):
    rmse_worst = y.max() - y.min()
    rmse_norm  = np.clip(rmse / rmse_worst, 0, 1)
    return 0.6 * max(r2, 0) + 0.4 * (1 - rmse_norm)

# Build combinations
combos = []
for ws in np.round(np.arange(WINDOW_MIN, WINDOW_MAX + 0.001, 0.01), 3):
    for w_start in np.round(np.arange(RANGE_START, RANGE_END - ws + 0.001, WINDOW_SLIDE), 3):
        w_end = round(w_start + ws, 3)
        if w_end > RANGE_END + 0.001:
            continue
        for n_pts in range(STEP_MIN, STEP_MAX + 1):
            combos.append((w_start, w_end, ws, n_pts))

total = len(combos)
print(f'Tong combinations: {total:,}')
print('Dang chay grid search...')

results = []
log_every = max(1, total // 20)

for i, (w_start, w_end, ws, n_pts) in enumerate(combos):
    mask = (V_array >= w_start - 0.001) & (V_array <= w_end + 0.001)
    V_win = V_array[mask]
    I_win = I_matrix[mask]

    if len(V_win) < 2:
        continue

    indices = np.unique(np.round(np.linspace(0, len(V_win)-1, n_pts)).astype(int))
    if len(indices) < 2:
        continue

    X = I_win[indices].T  # (10, n_features)
    if np.any(np.isnan(X)):
        continue

    r2, rmse, nc = eval_pls_loo(X, y)
    cs = composite_score(r2, rmse)

    results.append({
        'V_start'     : w_start,
        'V_end'       : w_end,
        'window_V'    : ws,
        'n_points'    : len(indices),
        'n_components': nc,
        'R2_LOO'      : round(r2,  5),
        'RMSE_LOO'    : round(rmse, 5),
        'composite'   : round(cs,  5),
        'V_selected'  : list(np.round(V_win[indices], 4))
    })

    if (i+1) % log_every == 0:
        pct = (i+1)/total*100
        best_r2_now = max(r['R2_LOO'] for r in results) if results else 0
        print(f'  {pct:5.1f}%  ({i+1:,}/{total:,})  Best R2 so far: {best_r2_now:.4f}')

results_df = pd.DataFrame(results).sort_values('composite', ascending=False).reset_index(drop=True)
results_df.to_csv('pls_grid_search_results.csv', index=False)
print(f'\nGrid search done! {len(results_df):,} valid models.')
print(f'Best: V=[{results_df.iloc[0]["V_start"]},{results_df.iloc[0]["V_end"]}]  '
      f'n={results_df.iloc[0]["n_points"]}pts  R2={results_df.iloc[0]["R2_LOO"]:.4f}  '
      f'RMSE={results_df.iloc[0]["RMSE_LOO"]:.4f}')

---
## 3. Top Models

In [ ]:
cols = ['V_start','V_end','window_V','n_points','n_components','R2_LOO','RMSE_LOO','composite']
print(f'=== TOP {TOP_N} MODELS ===')
print(results_df[cols].head(TOP_N).to_string(index=True))

---
## 4. Visualization Tong quan

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)
best = results_df.iloc[0]

# 4.1 Heatmap R2 theo V_start vs window_size
ax1 = fig.add_subplot(gs[0, :2])
pivot = results_df.groupby(['V_start','window_V'])['R2_LOO'].max().reset_index()
pt    = pivot.pivot(index='window_V', columns='V_start', values='R2_LOO')
im = ax1.imshow(pt.values, aspect='auto', origin='lower', cmap='RdYlGn',
                vmin=results_df['R2_LOO'].quantile(0.1), vmax=results_df['R2_LOO'].max())
plt.colorbar(im, ax=ax1, label='R2 (LOO-CV)')
xi = np.linspace(0, pt.shape[1]-1, 6).astype(int)
yi = np.linspace(0, pt.shape[0]-1, 6).astype(int)
ax1.set_xticks(xi); ax1.set_xticklabels([f'{v:.2f}' for v in pt.columns[xi]])
ax1.set_yticks(yi); ax1.set_yticklabels([f'{v:.2f}' for v in pt.index[yi]])
ax1.set_xlabel('V_start (V)', fontsize=11)
ax1.set_ylabel('Window size (V)', fontsize=11)
ax1.set_title(f'R2 Heatmap  |  Best: V=[{best["V_start"]},{best["V_end"]}]  '
              f'n={best["n_points"]}pts  R2={best["R2_LOO"]:.4f}', fontsize=11, fontweight='bold')

# 4.2 R2 distribution
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(results_df['R2_LOO'], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(best['R2_LOO'], color='red', linestyle='--', lw=2, label=f'Best={best["R2_LOO"]:.4f}')
ax2.axvline(results_df['R2_LOO'].median(), color='orange', linestyle='--', lw=1.5,
            label=f'Median={results_df["R2_LOO"].median():.4f}')
ax2.set_xlabel('R2 (LOO-CV)', fontsize=10); ax2.set_ylabel('Count', fontsize=10)
ax2.set_title('R2 Distribution', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# 4.3 R2 vs n_points
ax3 = fig.add_subplot(gs[1, 0])
g = results_df.groupby('n_points')['R2_LOO'].agg(['max','mean'])
ax3.plot(g.index, g['max'],  'ro-', lw=2, ms=7, label='Max R2')
ax3.plot(g.index, g['mean'], 'bs--', lw=1.5, ms=5, label='Mean R2')
ax3.set_xlabel('So diem (n_points)', fontsize=10); ax3.set_ylabel('R2 (LOO-CV)', fontsize=10)
ax3.set_title('R2 vs So diem features', fontsize=11, fontweight='bold')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

# 4.4 R2 vs window_size
ax4 = fig.add_subplot(gs[1, 1])
g2 = results_df.groupby('window_V')['R2_LOO'].agg(['max','mean'])
ax4.plot(g2.index, g2['max'],  'ro-', lw=1.5, ms=4, label='Max R2')
ax4.plot(g2.index, g2['mean'], 'bs--', lw=1.2, ms=3, label='Mean R2')
ax4.set_xlabel('Window size (V)', fontsize=10); ax4.set_ylabel('R2 (LOO-CV)', fontsize=10)
ax4.set_title('R2 vs Window size', fontsize=11, fontweight='bold')
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

# 4.5 Top 20 composite score
ax5 = fig.add_subplot(gs[1, 2])
top20  = results_df.head(20)
labels = [f'[{r.V_start:.2f},{r.V_end:.2f}]\n{r.n_points}pts' for _, r in top20.iterrows()]
ax5.barh(range(len(top20)), top20['composite'], color='steelblue', alpha=0.8)
ax5.set_yticks(range(len(top20))); ax5.set_yticklabels(labels, fontsize=6.5)
ax5.invert_yaxis()
ax5.set_xlabel('Composite Score', fontsize=10)
ax5.set_title('Top 20 Models', fontsize=11, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='x')

plt.savefig('pls_grid_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Detail: Best Model

In [ ]:
best = results_df.iloc[0]
print('=== BEST MODEL ===')
for k in ['V_start','V_end','window_V','n_points','n_components','R2_LOO','RMSE_LOO','composite']:
    print(f'  {k:<15}: {best[k]}')
print(f'  {"V_selected":<15}: {best["V_selected"]}')

# Rebuild
V_best = np.array(best['V_selected'])
X_best = np.array([I_matrix[np.argmin(np.abs(V_array - tv))] for tv in V_best]).T

scaler   = StandardScaler()
X_best_s = scaler.fit_transform(X_best)
pls_best = PLSRegression(n_components=int(best['n_components']))
pls_best.fit(X_best_s, y)
y_loo   = cross_val_predict(pls_best, X_best_s, y, cv=LeaveOneOut()).ravel()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CV curves
ax1 = axes[0]
lc = plt.cm.plasma(np.linspace(0.1, 0.9, len(CONCENTRATIONS)))
for i, (col, conc) in enumerate(zip(CONC_COLS, CONCENTRATIONS)):
    ax1.plot(df_fwd['V'], df_fwd[col], color=lc[i], label=f'{conc}mM', lw=1.3, alpha=0.8)
for v in V_best:
    ax1.axvline(x=v, color='red', lw=1.0, alpha=0.7, linestyle='--')
ax1.axvspan(best['V_start'], best['V_end'], alpha=0.08, color='red')
ax1.set_xlabel('Potential (V)', fontsize=10); ax1.set_ylabel('Current (uA)', fontsize=10)
ax1.set_title(f'CV Curves + Selected V\n[{best["V_start"]}->{best["V_end"]}V, {best["n_points"]}pts]', fontweight='bold')
ax1.legend(fontsize=6, ncol=2); ax1.grid(True, alpha=0.3)

# Predicted vs Actual
ax2 = axes[1]
ax2.scatter(y, y_loo, s=100, color='steelblue', zorder=5)
lim = [y.min()-0.4, y.max()+0.4]
ax2.plot(lim, lim, 'r--', lw=1.5)
for a, p in zip(y, y_loo):
    ax2.annotate(f'{a}mM', (a, p), textcoords='offset points', xytext=(5,4), fontsize=8)
ax2.set_xlabel('Actual (mM)', fontsize=10); ax2.set_ylabel('Predicted (mM)', fontsize=10)
ax2.set_title(f'Predicted vs Actual (LOO)\nR2={best["R2_LOO"]:.4f}  RMSE={best["RMSE_LOO"]:.4f}mM', fontweight='bold')
ax2.grid(True, alpha=0.3)

# Residuals
ax3 = axes[2]
res = y_loo - y
ax3.bar(y, res, width=0.08, color=['#e74c3c' if r < 0 else '#3498db' for r in res], alpha=0.85, edgecolor='black')
ax3.axhline(0, color='black', lw=1)
for yi, ri in zip(y, res):
    ax3.annotate(f'{ri:.2f}', (yi, ri), textcoords='offset points',
                 xytext=(0, 5 if ri>=0 else -13), ha='center', fontsize=8)
ax3.set_xlabel('Actual (mM)', fontsize=10); ax3.set_ylabel('Residual (mM)', fontsize=10)
ax3.set_title('Residuals (LOO-CV)', fontweight='bold'); ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Best PLS  V=[{best["V_start"]},{best["V_end"]}]  {best["n_points"]}pts  '
             f'nComp={best["n_components"]}  R2={best["R2_LOO"]:.4f}  RMSE={best["RMSE_LOO"]:.4f}mM',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('pls_best_model.png', dpi=150, bbox_inches='tight')
plt.show()

# Result table
print('\n=== Prediction Table ===')
df_res = pd.DataFrame({'Actual(mM)': y, 'Predicted_LOO(mM)': y_loo.round(4),
                       'Residual': (y_loo-y).round(4), 'AbsError': np.abs(y_loo-y).round(4)})
print(df_res.to_string(index=False))
print(f'\nMean Abs Error: {np.abs(y_loo-y).mean():.4f} mM')